# Kacchi Aloo: tickets S01–S19

This competition lets a team select up to 25 final submissions and the private leaderboard counts the best of them. These tickets were chosen in order of marginal value by E38 (`solution/portfolio_final.py`), which weighs the final selection over three label models of the noisy band edges (B's out-of-fold distance model E16, a parametric band, and edge-wise isotonic regression E30) by how well each explains our known public scores. Each ticket is a rule computed from the training data:

* **band(lo, hi)**: predict 1 when lo ≤ aloo/guest ≤ hi;
* **window**: the sharp band A with a block of rows flipped, taken from the uncertainty ranking of E16 (out-of-fold error by distance), E30 (isotonic edges) or E31 (Bayesian change-point).

No test row is labelled by hand. Run on Kaggle with the competition data attached; it writes one CSV per ticket.

In [1]:
import glob, hashlib, os, unicodedata
import numpy as np, pandas as pd

found = sorted(glob.glob("/kaggle/input/**/train.csv", recursive=True)) + [p for p in ("../data/train.csv", "data/train.csv") if os.path.exists(p)]
root = os.path.dirname(found[0])
tr_raw = pd.read_csv(f"{root}/train.csv", dtype=str)
te_raw = pd.read_csv(f"{root}/test.csv", dtype=str)


def to_number(v):
    """Float from a cell; any Unicode decimal digit (e.g. Bangla) becomes ASCII."""
    if pd.isna(v):
        return np.nan
    s = "".join(str(unicodedata.decimal(c)) if unicodedata.decimal(c, None) is not None else c
                for c in str(v).strip().replace(",", ""))
    try:
        return float(s)
    except ValueError:
        return np.nan


def add_apg(d):
    """aloo per guest; the few aloo counts typed x10 (ratio > 3) are divided by 10."""
    guests, aloo = d.guests.map(to_number), d.aloo_count.map(to_number)
    aloo = np.where(aloo / guests > 3.0, aloo / 10, aloo)
    return d.assign(apg=aloo / guests)


tr = add_apg(tr_raw).dropna(subset=["apg"]).reset_index(drop=True)   # 24 train rows have no aloo_count
tr["y"] = tr.went_back_for_seconds.astype(int)
te = add_apg(te_raw)
print(f"data: {root} | train {len(tr)} rows with aloo_count | test {len(te)} rows")

data: ../data | train 776 rows with aloo_count | test 400 rows


In [2]:
def midcut_cuts(r, y):
    """Each edge sits at the midpoint between neighbouring training ratios that maximises accuracy on its side of
    1.5 (median of tied optima)."""
    def best_cut(x, yy, side):
        xs = np.sort(np.unique(x))
        c = (xs[:-1] + xs[1:]) / 2
        accs = np.array([((x >= t) == yy).mean() if side == "lo" else ((x <= t) == yy).mean() for t in c])
        b = np.flatnonzero(accs == accs.max())
        return c[b[len(b) // 2]]
    m = r < 1.5
    return best_cut(r[m], y[m], "lo"), best_cut(r[~m], y[~m], "hi")


def band(x, lo, hi):
    return ((x >= lo) & (x <= hi)).astype(int)

In [3]:
from sklearn.model_selection import StratifiedKFold


def sharp_band_probabilities(tr, te, bins=(0.0, 0.01, 0.035, 0.1, 0.4, np.inf), repeats=10, seed=0):
    """P(y=1) per test row = the sharp band's out-of-fold error rate at that distance from the nearest edge.
    Test rows inside the training gap around an edge get P from a uniform prior on where the cut lies."""
    r, y = tr.apg.values, tr.y.values
    lo, hi = midcut_cuts(r, y)
    d_oof, e_oof = [], []
    for k in range(repeats):
        for t, v in StratifiedKFold(5, shuffle=True, random_state=seed + k).split(tr, y):
            l2, h2 = midcut_cuts(r[t], y[t])
            d_oof.append(np.minimum(abs(r[v] - l2), abs(r[v] - h2)))
            e_oof.append(band(r[v], l2, h2) != y[v])
    d_oof, e_oof = np.concatenate(d_oof), np.concatenate(e_oof)
    idx = np.digitize(d_oof, bins[1:-1])
    err = np.array([(e_oof[idx == b].sum() + 0.5) / ((idx == b).sum() + 1.0) for b in range(len(bins) - 1)])
    below_lo, above_lo = r[r < lo].max(), r[(r >= lo) & (r < 1.5)].min()
    below_hi, above_hi = r[(r <= hi) & (r > 1.5)].max(), r[r > hi].min()
    x = te.apg.values
    d = np.minimum(abs(x - lo), abs(x - hi))
    e = err[np.digitize(d, bins[1:-1])]
    P = np.where(band(x, lo, hi) == 1, 1 - e, e)
    q_lo = np.clip((x - below_lo) / (above_lo - below_lo), 0, 1)
    q_hi = np.clip((above_hi - x) / (above_hi - below_hi), 0, 1)
    g_lo, g_hi = (x > below_lo) & (x < above_lo), (x > below_hi) & (x < above_hi)
    P = np.where(g_lo, q_lo * (1 - err[0]) + (1 - q_lo) * err[0], P)
    P = np.where(g_hi, q_hi * (1 - err[0]) + (1 - q_hi) * err[0], P)
    return P, (lo, hi), err


P, (LO, HI), err = sharp_band_probabilities(tr, te)
print(f"cut-points {LO:.5f} / {HI:.5f}; out-of-fold error rate by distance to the edge "
      f"(0-0.01, 0.01-0.035, 0.035-0.1, 0.1-0.4, >0.4): {np.round(err, 3)}")

cut-points 0.97068 / 2.00983; out-of-fold error rate by distance to the edge (0-0.01, 0.01-0.035, 0.035-0.1, 0.1-0.4, >0.4): [0.127 0.374 0.148 0.022 0.   ]


In [4]:
from scipy.special import betaln
from sklearn.isotonic import IsotonicRegression

r_tr, y_tr, x_te = tr.apg.values, tr.y.values.astype(float), te.apg.values


def plateaus(r, y, lo_win, hi_win):
    mid = (r > lo_win[1]) & (r < hi_win[0])
    out = (r < lo_win[0]) | (r > hi_win[1])
    jef = lambda m: (y[m].sum() + 0.5) / (m.sum() + 1.0)
    return jef(mid), jef(out)


def isotonic_edges(r, y, x, lo_win=(0.85, 1.15), hi_win=(1.80, 2.20)):
    """E30: increasing isotonic fit on the lower-edge window, decreasing on the upper, plateau rates elsewhere."""
    p_mid, p_out = plateaus(r, y, lo_win, hi_win)
    wl = (r >= lo_win[0]) & (r <= lo_win[1]); wh = (r >= hi_win[0]) & (r <= hi_win[1])
    iso_l = IsotonicRegression(increasing=True, y_min=0, y_max=1, out_of_bounds="clip").fit(r[wl], y[wl])
    iso_h = IsotonicRegression(increasing=False, y_min=0, y_max=1, out_of_bounds="clip").fit(r[wh], y[wh])
    p = np.where((x > lo_win[1]) & (x < hi_win[0]), p_mid, p_out)
    p = np.where((x >= lo_win[0]) & (x <= lo_win[1]), iso_l.predict(np.clip(x, *lo_win)), p)
    return np.where((x >= hi_win[0]) & (x <= hi_win[1]), iso_h.predict(np.clip(x, *hi_win)), p)


def changepoint_edge(r, y, win, x, a0=1.0, b0=1.0):
    """E31, one edge: cut uniform on the window, a constant rate on each side (Beta prior), cut integrated out."""
    m = (r >= win[0]) & (r <= win[1])
    rr, yy = r[m], y[m]
    u = np.unique(np.r_[win[0], rr, win[1]])
    lo_k, hi_k = u[:-1], u[1:]
    above = rr[None, :] >= hi_k[:, None]
    n_a, k_a = above.sum(1), (above * yy[None, :]).sum(1)
    n_b, k_b = len(rr) - n_a, yy.sum() - k_a
    ll = (betaln(a0 + k_a, b0 + n_a - k_a) + betaln(a0 + k_b, b0 + n_b - k_b) - 2 * betaln(a0, b0)
          + np.log(np.maximum(hi_k - lo_k, 1e-12)))
    post = np.exp(ll - ll.max()); post /= post.sum()
    th_a, th_b = (a0 + k_a) / (a0 + b0 + n_a), (a0 + k_b) / (a0 + b0 + n_b)
    q = np.clip((x[None, :] - lo_k[:, None]) / (hi_k - lo_k)[:, None], 0, 1)
    return post @ (q * th_a[:, None] + (1 - q) * th_b[:, None])


def changepoint_edges(r, y, x, lo_win=(0.85, 1.15), hi_win=(1.80, 2.25)):
    p_mid, p_out = plateaus(r, y, lo_win, hi_win)
    p = np.where((x > lo_win[1]) & (x < hi_win[0]), p_mid, p_out)
    il = (x >= lo_win[0]) & (x <= lo_win[1]); ih = (x >= hi_win[0]) & (x <= hi_win[1])
    p[il] = changepoint_edge(r, y, lo_win, x[il])
    p[ih] = changepoint_edge(r, y, hi_win, x[ih])
    return p


P30 = isotonic_edges(r_tr, y_tr, x_te)
P31 = changepoint_edges(r_tr, y_tr, x_te)
RANKING_P = {"E16": P, "E30": P30, "E31": P31}
print("rows with 0.25 < P < 0.75:", {k: int(((v > 0.25) & (v < 0.75)).sum()) for k, v in RANKING_P.items()})

rows with 0.25 < P < 0.75: {'E16': 26, 'E30': 17, 'E31': 3}


In [5]:
SPECS = {
 "S01": {
  "family": "window",
  "ranking": "E16",
  "side": "both",
  "start": 6,
  "width": 5
 },
 "S02": {
  "family": "band",
  "lo": 0.95109,
  "hi": 2.01193
 },
 "S03": {
  "family": "window",
  "ranking": "E16",
  "side": "both",
  "start": 4,
  "width": 3
 },
 "S04": {
  "family": "window",
  "ranking": "E16",
  "side": "both",
  "start": 13,
  "width": 6
 },
 "S05": {
  "family": "window",
  "ranking": "E31",
  "side": "upper",
  "start": 0,
  "width": 5
 },
 "S06": {
  "family": "window",
  "ranking": "E16",
  "side": "both",
  "start": 0,
  "width": 4
 },
 "S07": {
  "family": "window",
  "ranking": "E16",
  "side": "lower",
  "start": 0,
  "width": 8
 },
 "S08": {
  "family": "window",
  "ranking": "E30",
  "side": "both",
  "start": 11,
  "width": 5
 },
 "S09": {
  "family": "window",
  "ranking": "E16",
  "side": "both",
  "start": 18,
  "width": 4
 },
 "S10": {
  "family": "window",
  "ranking": "E16",
  "side": "both",
  "start": 11,
  "width": 4
 },
 "S11": {
  "family": "band",
  "lo": 0.96938,
  "hi": 1.98867
 },
 "S12": {
  "family": "window",
  "ranking": "E16",
  "side": "upper",
  "start": 16,
  "width": 5
 },
 "S13": {
  "family": "window",
  "ranking": "E30",
  "side": "both",
  "start": 19,
  "width": 4
 },
 "S14": {
  "family": "band",
  "lo": 1.00347,
  "hi": 1.98378
 },
 "S15": {
  "family": "band",
  "lo": 0.96462,
  "hi": 2.01429
 },
 "S16": {
  "family": "window",
  "ranking": "E31",
  "side": "upper",
  "start": 3,
  "width": 4
 },
 "S17": {
  "family": "window",
  "ranking": "E16",
  "side": "upper",
  "start": 12,
  "width": 4
 },
 "S18": {
  "family": "window",
  "ranking": "E16",
  "side": "both",
  "start": 6,
  "width": 3
 },
 "S19": {
  "family": "window",
  "ranking": "E16",
  "side": "both",
  "start": 15,
  "width": 3
 }
}
EXPECTED = {
 "S01": "8967009dcf2a40f37f18c89b0ab28cff",
 "S02": "23ea9afa1fd82d4344add28cf389c923",
 "S03": "29867c414ec51695a49d7ce26b7155ae",
 "S04": "de9ce4ecdb83a533ce36c98d9f89b8af",
 "S05": "8e62a2df371dd0f475bc41a32c093500",
 "S06": "71af6f31922a7cf3f82c7b6940ec718e",
 "S07": "29eb58a0c81772f651ab0fcf0dcf64bc",
 "S08": "b67657d5ba690419acd197ebe047db34",
 "S09": "aee704ac71975af1b0e734dcf61c4895",
 "S10": "3cfce5849a1997b9d802d47a29eeed45",
 "S11": "9b1a7ee4abb1c21341c9801a527ac0bf",
 "S12": "34de600f56882c1abfb548f9cb17d213",
 "S13": "a764f84aebfab0474e844057b2d4a427",
 "S14": "049e982aad0d79d81f3fec729c66b5ee",
 "S15": "71a217f36f9d0a276b1e4307627a65d8",
 "S16": "e7c0df502ec0b4c182ca78b7e629a7b9",
 "S17": "7bd21bc53e03f783523eb58360379c5b",
 "S18": "71aafa0f88cc82509e561b4e1fbef15b",
 "S19": "5d1a5b8f32a7ff0bbc4f4255a1b6bf46"
}

A = band(x_te, LO, HI)
d_edge = np.minimum(abs(x_te - LO), abs(x_te - HI))
low = x_te < 1.5


def ticket(spec):
    if spec["family"] == "band":
        return band(x_te, spec["lo"], spec["hi"])
    order = np.lexsort((d_edge, np.abs(RANKING_P[spec["ranking"]] - 0.5)))
    o = {"both": order, "lower": order[low[order]], "upper": order[~low[order]]}[spec["side"]]
    v = A.copy()
    v[o[spec["start"]:spec["start"] + spec["width"]]] ^= 1
    return v


def describe(spec):
    if spec["family"] == "band":
        return f"band {spec['lo']} <= aloo/guest <= {spec['hi']}"
    edge = {"both": "both edges", "lower": "lower edge", "upper": "upper edge"}[spec["side"]]
    return f"A + flip {spec['ranking']} uncertainty ranks {spec['start'] + 1}-{spec['start'] + spec['width']} ({edge})"


rows = []
for name, spec in SPECS.items():
    v = ticket(spec)
    sub = pd.DataFrame({"wedding_id": te.wedding_id, "went_back_for_seconds": v.astype(int)})
    sub.to_csv(f"{name}.csv", index=False)
    fp = hashlib.md5("".join(f"{i}{p}" for i, p in zip(sub.wedding_id, sub.went_back_for_seconds)).encode()).hexdigest()
    rows.append(dict(file=f"{name}.csv", rule=describe(spec), rows_vs_A=int((v != A).sum()), ones=int(v.sum()),
                     matches_plan=fp == EXPECTED[name]))
summary = pd.DataFrame(rows)
print(summary.to_string(index=False))
print("all tickets match the optimizer's files:", bool(summary.matches_plan.all()))

   file                                              rule  rows_vs_A  ones  matches_plan
S01.csv  A + flip E16 uncertainty ranks 7-11 (both edges)          5   183          True
S02.csv             band 0.95109 <= aloo/guest <= 2.01193          6   188          True
S03.csv   A + flip E16 uncertainty ranks 5-7 (both edges)          3   183          True
S04.csv A + flip E16 uncertainty ranks 14-19 (both edges)          6   184          True
S05.csv   A + flip E31 uncertainty ranks 1-5 (upper edge)          5   179          True
S06.csv   A + flip E16 uncertainty ranks 1-4 (both edges)          4   184          True
S07.csv   A + flip E16 uncertainty ranks 1-8 (lower edge)          8   184          True
S08.csv A + flip E30 uncertainty ranks 12-16 (both edges)          5   177          True
S09.csv A + flip E16 uncertainty ranks 19-22 (both edges)          4   178          True
S10.csv A + flip E16 uncertainty ranks 12-15 (both edges)          4   184          True
S11.csv             b